# Export Data for CA_zoom.jpg and Unburned_delta_time_comparison.jpg
Loads and clips all model output rasters, then saves to `../figure_data/figure_CA_zoom_and_unburned_delta/`.

In [1]:
import os

import numpy as np
import xarray as xr

from conus_biomass import dir_info
from conus_biomass.make_figures import maps

OUT_DIR = "../figure_data/figure_S_CA_zoom_and_unburned_delta/"
os.makedirs(OUT_DIR, exist_ok=True)

## Helper functions

In [2]:
def get_filtered_output(
    year,
    fdir=dir_info.dir_processed + "model_results/1000m_2026feb02_processed/",
    ftype="netcdf",
    vartype="predicted_biomass_filtered_",
):
    fname = fdir + vartype + str(year)
    if ftype == "zarr":
        ds = xr.open_zarr(fname + ".zarr")
    elif ftype == "netcdf":
        ds = xr.load_dataset(fname + ".nc")
    return ds


def clip_to_shape(da, shp):
    if shp.crs != da.rio.crs:
        shp = shp.to_crs(da.rio.crs)
    return da.rio.clip(shp.geometry, shp.crs, drop=True)

In [3]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

years = np.arange(2005, 2023)

## Load and clip unburned gains

In [4]:
unburned_gains_west = []
for year in years:
    print(year)
    ds = get_filtered_output(year=year, vartype="predicted_biomass_delta_unburned_filtered_")
    ds = ds.rio.write_crs(crs)
    da = clip_to_shape(da=ds["predicted_biomass_delta_unburned"], shp=maps.SHP_WESTERN)
    da = da.assign_coords(year=year)
    unburned_gains_west.append(da)
    del ds

unburned_gains_west = xr.concat(unburned_gains_west, dim="year")

2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022


## Save

In [6]:
unburned_gains_west.to_dataset(name="predicted_biomass_delta_unburned").to_netcdf(
    OUT_DIR + "unburned_gains_west.nc"
)
print("Saved unburned_gains_west.nc")

Saved unburned_gains_west.nc
